## HW 4

### Описание
В скором времени AI агенты достигнут уровня развития человека. А значит им придется столкнуться с проблемами, которые раньше были свойственны только людям.
Им придется сдавать ЕГЭ. Поэтому мы решили заранее помочь им подготовиться к экзамену по литературе.
Агент прочитал романы Война и Мир Толстого и Преступление и наказание Достоевского, но мало что запомнил. Поможем ему понять эти романы лучше.
Для этого мы научим его определять по вырванной цитате из романа, к какому роману она относится.

### Данные
train.csv - обучающая выборка, содержит 2 поля:
* text - текст цитаты
* label - метка класса (0 - Война и Мир, 1 - Преступление и наказание)

mlm.txt - plain текст вперемешку цитат из обоих романов.

### Задача
1. **Обучить классификатор** (фактически та же задача, что и на практике 5):
* Разбить train.csv на train и valid выборки
* Обучить модель, которая по тексту цитаты будет определять к какому роману она относится. Необходимо использовать модель "distilbert/distilroberta-base" из huggingface.
* Протестировать на valid: confusion_matrix, accuracy, f1. Сохранить эти метрики.

In [ ]:
from datasets import load_dataset, Dataset, concatenate_datasets
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix
from transformers import AutoTokenizer, AutoModelForSequenceClassification, TrainingArguments, Trainer
import numpy as np, pandas as pd
import math
import re

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
def compute_metrics(p):
    preds = np.argmax(p.predictions, axis=1)
    true_labels = p.label_ids
    return {"accuracy": accuracy_score(true_labels, preds),
            "f1": f1_score(true_labels, preds, average="weighted")}
            #"confusion_matrix": confusion_matrix(true_labels, preds).tolist()}

In [ ]:
data_path = '/content/drive/MyDrive/DL_HSE_2025_fall/homeworks/hw4/data/'

In [ ]:
ds = load_dataset("csv", data_files=data_path+"train.csv", encoding='windows-1252')["train"].train_test_split(test_size=0.2, seed=42) #, stratify_by_column="label")

Generating train split: 0 examples [00:00, ? examples/s]

In [ ]:
model_name = "distilbert/distilroberta-base"
tok = AutoTokenizer.from_pretrained(model_name)
ds = ds.map(lambda x: tok(x["text"], padding="max_length", truncation=True), batched=True) # max_length

Map:   0%|          | 0/4512 [00:00<?, ? examples/s]

Map:   0%|          | 0/1128 [00:00<?, ? examples/s]

In [ ]:
# классификационная модель
model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=2)
args = TrainingArguments(output_dir=data_path+"base_classifier_out",
                         num_train_epochs=5,
                         learning_rate=2e-5,
                         weight_decay=0.01,
                         per_device_train_batch_size=16,
                         per_device_eval_batch_size=16,
                         eval_strategy="epoch",
                         save_strategy="epoch",
                         load_best_model_at_end=True,
                         metric_for_best_model="accuracy",
                         logging_steps=100)
trainer = Trainer(model=model,
                  args=args,
                  train_dataset=ds["train"],
                  eval_dataset=ds["test"],
                  tokenizer=tok,
                  compute_metrics=compute_metrics)

Some weights of RobertaForSequenceClassification were not initialized from the model checkpoint at distilbert/distilroberta-base and are newly initialized: ['classifier.dense.bias', 'classifier.dense.weight', 'classifier.out_proj.bias', 'classifier.out_proj.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
/tmp/ipython-input-204644475.py:14: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(model=model,


In [ ]:
'''
pred = trainer.predict(ds["test"])
preds = np.argmax(pred.predictions, axis=1)
labels = pred.label_ids
cm = confusion_matrix(labels, preds)
pd.DataFrame([{"accuracy": accuracy_score(labels, preds),
               "f1": f1_score(labels, preds, average="weighted"),
               "confusion_matrix": cm.tolist()}]).to_csv("metrics.csv", index=False)
trainer.save_model("model")
'''

In [ ]:
before = trainer.evaluate()
print("Before training:")
print(f"Loss: {before['eval_loss']:.2f}")
print(f"Perplexity: {math.exp(before['eval_loss']):.2f}")
print(before)

Before training:
Loss: 0.71
Perplexity: 2.03
{'eval_loss': 0.7060303092002869, 'eval_model_preparation_time': 0.0017, 'eval_accuracy': 0.5150709219858156, 'eval_f1': 0.3502120604725089, 'eval_runtime': 15.9156, 'eval_samples_per_second': 70.874, 'eval_steps_per_second': 4.461}


До:

accuracy 0.518

f1       0.449

In [ ]:
trainer.train()

Epoch,Training Loss,Validation Loss,Model Preparation Time,Accuracy,F1
1,0.266400,0.262501,0.001700,0.875887,0.874833
2,0.226700,0.247480,0.001700,0.867021,0.866800
3,0.210300,0.228472,0.001700,0.888298,0.888321
4,0.215900,0.217441,0.001700,0.891844,0.891844
5,0.162500,0.221717,0.001700,0.890957,0.890933


TrainOutput(global_step=1410, training_loss=0.2330255189685957, metrics={'train_runtime': 1158.2614, 'train_samples_per_second': 19.477, 'train_steps_per_second': 1.217, 'total_flos': 2988464513679360.0, 'train_loss': 0.2330255189685957, 'epoch': 5.0})

In [ ]:
after = trainer.evaluate()
print("After training:")
print(f"Loss: {after['eval_loss']:.2f}")
print(f"Perplexity: {math.exp(after['eval_loss']):.2f}")
print(after)

After training:
Loss: 0.22
Perplexity: 1.24
{'eval_loss': 0.21744118630886078, 'eval_model_preparation_time': 0.0017, 'eval_accuracy': 0.8918439716312057, 'eval_f1': 0.8918439716312057, 'eval_runtime': 16.8464, 'eval_samples_per_second': 66.958, 'eval_steps_per_second': 4.215, 'epoch': 5.0}


После:

accuracy 0.892

f1       0.892

In [ ]:
drive_model_path = '/content/drive/MyDrive/DL_HSE_2025_fall/homeworks/hw4/data/base_classifier'
trainer.save_model(drive_model_path)
#tok.save_pretrained(drive_model_path)

2. **Претренировать модель с помощью unsupervised masked language modeling** на train-test.txt
* Воспользоваться вот этим туториалом https://huggingface.co/docs/transformers/main/tasks/masked_language_modeling
* Вывести метрики perplexity и loss до и после обучения

In [ ]:
from datasets import load_dataset, Dataset
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix
from transformers import AutoTokenizer, AutoModelForMaskedLM, DataCollatorForLanguageModeling, TrainingArguments, Trainer
import numpy as np, pandas as pd
import math
import re
import torch

In [ ]:
def split_text(text, max_len=65):
    parts = re.findall(r'.+?(?:[\.!\?…]|,|—|:;\"\'\)\]]+)(?=\s|$)', text, flags=re.S)
    rest = text
    for p in parts:
      rest = rest.replace(p, '', 1)
    if rest.strip():
      parts.append(rest.strip())
    chunks, cur = [], ""
    for p in (s.strip() for s in parts if s.strip()):
        if len(p) <= max_len:
            if cur and len(cur) + 1 + len(p) <= max_len:
                cur += " " + p
            else:
                if cur:
                  chunks.append(cur)
                cur = p
        else:
            if cur:
              chunks.append(cur)
    if cur:
      chunks.append(cur)
    return chunks

In [ ]:
raw = load_dataset("text", data_files=data_path+"mlm.txt")["train"]['text'][0]

Generating train split: 0 examples [00:00, ? examples/s]

In [ ]:
raw = split_text(raw, 65)

In [ ]:
raw = Dataset.from_dict({"text": raw})

In [ ]:
raw

Dataset({
    features: ['text'],
    num_rows: 43992
})

In [ ]:
raw = raw.train_test_split(test_size=0.2, seed=42)

In [ ]:
raw

DatasetDict({
    train: Dataset({
        features: ['text'],
        num_rows: 35193
    })
    test: Dataset({
        features: ['text'],
        num_rows: 8799
    })
})

In [ ]:
model_name = "distilbert/distilroberta-base"
tok = AutoTokenizer.from_pretrained(model_name)
lm_dataset = raw.map(lambda x: tok(x["text"]), batched=True, remove_columns=["text"])

'''
block_size=128
def group_texts(examples):
    # Concatenate all texts.
    concatenated_examples = {k: sum(examples[k], []) for k in examples.keys()}
    total_length = len(concatenated_examples[list(examples.keys())[0]])
    # We drop the small remainder, we could add padding if the model supported it instead of this drop, you can
    # customize this part to your needs.
    if total_length >= block_size:
        total_length = (total_length // block_size) * block_size
    # Split by chunks of block_size.
    result = {
        k: [t[i : i + block_size] for i in range(0, total_length, block_size)]
        for k, t in concatenated_examples.items()
    }
    return result

lm_dataset = lm_dataset.map(group_texts, batched=True)
'''
tok.pad_token = tok.eos_token
data_collator = DataCollatorForLanguageModeling(tok, mlm_probability=0.15)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/480 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Map:   0%|          | 0/35193 [00:00<?, ? examples/s]

Map:   0%|          | 0/8799 [00:00<?, ? examples/s]

In [ ]:
torch.cuda.empty_cache()

In [ ]:
# MLM модель
model = AutoModelForMaskedLM.from_pretrained(model_name)

args = TrainingArguments(
    output_dir=data_path+"mlm",
    num_train_epochs=5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    learning_rate=2e-5,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    save_total_limit=1,
    report_to="none"
)

trainer = Trainer(
    model=model,
    args=args,
    train_dataset=lm_dataset["train"],
    eval_dataset=lm_dataset["test"],
    data_collator=data_collator,
    tokenizer=tok
)

Some weights of the model checkpoint at distilbert/distilroberta-base were not used when initializing RobertaForMaskedLM: ['roberta.pooler.dense.bias', 'roberta.pooler.dense.weight']
- This IS expected if you are initializing RobertaForMaskedLM from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing RobertaForMaskedLM from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).
/tmp/ipython-input-3951847965.py:64: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(


In [ ]:
before = trainer.evaluate()
print("Before training:")
print(f"Loss: {before['eval_loss']:.2f}")
print(f"Perplexity: {math.exp(before['eval_loss']):.2f}")
print(before)

Before training:
Loss: 1.48
Perplexity: 4.41
{'eval_loss': 1.4830831289291382, 'eval_model_preparation_time': 0.0042, 'eval_runtime': 40.5994, 'eval_samples_per_second': 216.727, 'eval_steps_per_second': 13.547}


In [ ]:
trainer.train()

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'pad_token_id': 2}.


Epoch,Training Loss,Validation Loss,Model Preparation Time
1,1.054400,0.923523,0.004200
2,0.923300,0.803092,0.004200
3,0.843400,0.742237,0.004200
4,0.805500,0.696997,0.004200
5,0.772200,0.678776,0.004200


There were missing keys in the checkpoint model loaded: ['lm_head.decoder.weight', 'lm_head.decoder.bias'].


TrainOutput(global_step=11000, training_loss=0.9092751797762784, metrics={'train_runtime': 1222.0847, 'train_samples_per_second': 143.988, 'train_steps_per_second': 9.001, 'total_flos': 3272289733693728.0, 'train_loss': 0.9092751797762784, 'epoch': 5.0})

In [ ]:
after = trainer.evaluate()
print("After training:")
print(f"Loss: {after['eval_loss']:.2f}")
print(f"Perplexity: {math.exp(after['eval_loss']):.2f}")

After training:
Loss: 0.69
Perplexity: 2.00


In [ ]:
after

{'eval_loss': 0.6942470669746399,
 'eval_model_preparation_time': 0.0042,
 'eval_runtime': 20.844,
 'eval_samples_per_second': 422.135,
 'eval_steps_per_second': 26.386,
 'epoch': 5.0}

In [ ]:
drive_model_path = '/content/drive/MyDrive/DL_HSE_2025_fall/homeworks/hw4/data/mlm_model'
trainer.save_model(drive_model_path)
#tok.save_pretrained(drive_model_path)

In [ ]:
trainer.save_model('mlm_model')

In [ ]:
!zip -r mlm_model.zip mlm_model

updating: mlm_model/ (stored 0%)
updating: mlm_model/config.json (deflated 49%)
updating: mlm_model/model.safetensors (deflated 7%)
updating: mlm_model/training_args.bin (deflated 53%)
updating: mlm_model/vocab.json (deflated 59%)
updating: mlm_model/special_tokens_map.json (deflated 53%)
updating: mlm_model/tokenizer_config.json (deflated 75%)
updating: mlm_model/merges.txt (deflated 53%)
updating: mlm_model/tokenizer.json (deflated 82%)


In [ ]:
from google.colab import files
files.download("mlm_model.zip")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

3. **Перетренировать классификатор из пункта 1**, но использовать претренированные веса из пункта 2
* сравнить метрики с пунктом 1

In [ ]:
full = concatenate_datasets([ds["train"], ds["test"]])

In [ ]:
full

Dataset({
    features: ['text', 'label', 'input_ids', 'attention_mask'],
    num_rows: 5640
})

In [ ]:
classifier_with_mlm = AutoModelForSequenceClassification.from_pretrained("mlm_model", num_labels=2)
args_with_mlm = TrainingArguments(output_dir=data_path+"classifier_with_mlm_out",
                         num_train_epochs=5,
                         learning_rate=2e-5,
                         weight_decay=0.01,
                         per_device_train_batch_size=16,
                         per_device_eval_batch_size=16,
                         eval_strategy="epoch",
                         save_strategy="epoch",
                         load_best_model_at_end=True,
                         metric_for_best_model="accuracy",
                         logging_steps=100)

final_trainer = Trainer(model=classifier_with_mlm,
                  args=args_with_mlm,
                  train_dataset=full,
                  eval_dataset=ds["test"],
                  compute_metrics=compute_metrics,
                  tokenizer=tok)

Some weights of RobertaForSequenceClassification were not initialized from the model checkpoint at mlm_model and are newly initialized: ['classifier.dense.bias', 'classifier.dense.weight', 'classifier.out_proj.bias', 'classifier.out_proj.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
/tmp/ipython-input-566224785.py:14: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  final_trainer = Trainer(model=classifier_with_mlm,


In [ ]:
before = final_trainer.evaluate()
print("Before training:")
print(f"Loss: {before['eval_loss']:.2f}")
print(f"Perplexity: {math.exp(before['eval_loss']):.2f}")
print(before)

Before training:
Loss: 0.69
Perplexity: 2.00
{'eval_loss': 0.6934792995452881, 'eval_model_preparation_time': 0.009, 'eval_accuracy': 0.5150709219858156, 'eval_f1': 0.3502120604725089, 'eval_runtime': 8.5395, 'eval_samples_per_second': 132.092, 'eval_steps_per_second': 8.314}


In [ ]:
final_trainer.train()

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'pad_token_id': 1}.


Epoch,Training Loss,Validation Loss,Model Preparation Time,Accuracy,F1
1,0.233900,0.209905,0.009000,0.881206,0.881217
2,0.204200,0.181676,0.009000,0.901596,0.901096
3,0.174900,0.161530,0.009000,0.922872,0.922849
4,0.172900,0.145728,0.009000,0.931738,0.931752
5,0.141200,0.134915,0.009000,0.940603,0.940615


TrainOutput(global_step=1765, training_loss=0.19537426945845737, metrics={'train_runtime': 571.7334, 'train_samples_per_second': 49.324, 'train_steps_per_second': 3.087, 'total_flos': 3735580642099200.0, 'train_loss': 0.19537426945845737, 'epoch': 5.0})

In [ ]:
drive_model_path = '/content/drive/MyDrive/DL_HSE_2025_fall/homeworks/hw4/data/classifier_with_mlm'
final_trainer.save_model(drive_model_path)
#tok.save_pretrained(drive_model_path)

In [ ]:
final_trainer.save_model('classifier_with_mlm')

In [ ]:
!zip -r classifier_with_mlm.zip classifier_with_mlm

  adding: classifier_with_mlm/ (stored 0%)
  adding: classifier_with_mlm/config.json (deflated 50%)
  adding: classifier_with_mlm/model.safetensors (deflated 7%)
  adding: classifier_with_mlm/training_args.bin (deflated 53%)
  adding: classifier_with_mlm/vocab.json (deflated 59%)
  adding: classifier_with_mlm/special_tokens_map.json (deflated 52%)
  adding: classifier_with_mlm/tokenizer_config.json (deflated 75%)
  adding: classifier_with_mlm/merges.txt (deflated 53%)
  adding: classifier_with_mlm/tokenizer.json (deflated 82%)


In [ ]:
from google.colab import files
files.download("classifier_with_mlm.zip")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
after = final_trainer.evaluate()
print("After training:")
print(f"Loss: {after['eval_loss']:.2f}")
print(f"Perplexity: {math.exp(after['eval_loss']):.2f}")

After training:
Loss: 0.13
Perplexity: 1.14


In [ ]:
after

In [ ]:
df = pd.read_csv(data_path+"submission.csv", encoding='windows-1252')
texts = df["text"].tolist()
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
tok = AutoTokenizer.from_pretrained("./classifier_with_mlm")
model = AutoModelForSequenceClassification.from_pretrained("./classifier_with_mlm").to(device)
labels = []

for t in texts:
    inputs = tok(t, return_tensors="pt", padding=True, truncation=True).to(device)
    with torch.no_grad():
        logits = model(**inputs).logits
    labels.append(int(logits.argmax(dim=1).cpu().item()))

pd.DataFrame({"text": texts, "label": labels}).to_csv(data_path+"submission_final.csv", index=False)


In [ ]:
texts_with_quotes = [f'"{t}"' for t in texts]
pd.DataFrame({"text": texts_with_quotes, "label": labels}).to_csv(data_path+"submission_final.csv", index=False, encoding='windows-1252')